### چه کالاهایی در چه زمان هایی پیک خرید رو میزنند؟
### توزیع فروش کالاها در هر دسته بندی چه ایده ای به ما میتونه بده؟
### با مقایسه فروش یک نوع کالا در یک دسته بندی آیا می تونیم فروشنده های برتر رو پیدا کنیم؟
### آیا کامنت ها با میزان فروش همبستگی دارند و میشه ازشون یک متریک ساخت ؟
### میزان رضایت از کالا را هم با امتیاز و هم با کامنت ها متریک کنیم؟

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

In [ ]:
data_path = os.getenv('DATA_PATH')
products = pd.read_csv(f'{data_path}/BaSalam.products.csv')
reviews = pd.read_csv(f'{data_path}/BaSalam.reviews.csv')
labeled_comments = pd.read_csv(f"{data_path}/all_labeled_comments.csv")
map_category = pd.read_csv(f"{data_path}/category_map.csv")
category_mapping = dict(zip(map_category['sub_category'], map_category['main_category']))
products['main_category'] = products['categoryTitle'].map(category_mapping).fillna("سایر")
products[['categoryTitle','main_category']].sample()

# 1

- no data

# 2

In [ ]:
products['main_category'].value_counts()

In [ ]:
fig = px.histogram(products[products['sales_count_week']!= 0], x='main_category', y='sales_count_week',
              title='Distribution of Weekly Sales Count by Main Category',
              hover_data=['name', 'price'])
fig.update_layout(
    xaxis_title='Main Category',
    yaxis_title='Weekly Sales Count',
    xaxis={'categoryorder':'total descending'},
    height=600,
    width=1000
)
fig.show()

In [ ]:
products['sales_count_week']

In [ ]:
for category in products['main_category'].unique().tolist():
    fig = px.histogram(products[products['main_category']==category], x='sales_count_week', title=f'Histogram of {category}')
    fig.update_layout(xaxis_title='sales count week', yaxis_title='Count', xaxis={'categoryorder':'total descending'})
    fig.show()

In [ ]:
for category in products['main_category'].unique().tolist():
    fig = px.histogram(products[(products['main_category']==category) & (products['sales_count_week'] != 0)], x='sales_count_week', title=f'Histogram of {category}')
    fig.update_layout(xaxis_title='sales count week', yaxis_title='Count', xaxis={'categoryorder':'total descending'})
    fig.show()

# 3

In [ ]:
x = products[products['name'].notna()]
x[(x['name'].str.contains('پسته'))&(x['main_category'].isin(['مواد غذایی']))]

# 4

In [ ]:
count_comments_by_product = reviews[reviews['star'] > 3].groupby('productId').size().reset_index()
count_comments_by_product.columns = ['productId', 'count']
count_comments_by_product = pd.merge(count_comments_by_product, products[['_id', 'sales_count_week']], left_on='productId', right_on='_id')
count_comments_by_product

In [ ]:
count_comments_by_product[['count', 'sales_count_week']].corr()

# 5

In [ ]:
rating_by_product = reviews.groupby('productId')['star'].agg(['count', 'mean']).sort_values(
        by=['mean', 'count'], ascending=[False, False]).reset_index()
rating_by_product

In [ ]:
comments_by_product = labeled_comments.copy()
comments_by_product['sentiment_by_parsbert'] = comments_by_product['sentiment_by_parsbert'].map({
    'Positive': 5,
    'Negative': 0
})
comments_by_product = pd.merge(comments_by_product, reviews[['_id', 'productId']], left_on='_id', right_on='_id', how='left')
comments_by_product = comments_by_product.groupby('productId')['sentiment_by_parsbert'].agg(['mean', 'count']).reset_index()
comments_by_product

In [ ]:
products_metric = []
for id in rating_by_product['productId'].sample(10).tolist():
    if id in comments_by_product['productId'].tolist():
        try:
            c_m = comments_by_product[comments_by_product['productId']==id]['mean'].values.tolist()[0]
            c_c = comments_by_product[comments_by_product['productId']==id]['count'].values.tolist()[0]
        except:
            print(id)
            break
    else:
        c_m = 0
        c_c = 0
    r_m = rating_by_product[rating_by_product['productId']==id]['mean'].values.tolist()[0]
    r_c = rating_by_product[rating_by_product['productId']==id]['count'].values.tolist()[0]

    metric = (r_m*r_c + c_m*c_c ) / (c_c + r_c)
    print(r_m, r_c,  c_m, c_c)
    print((r_m*r_c + c_m*c_c ) , (c_c + r_c))
    print(metric)
    print()
    products_metric.append(metric)